# Exporter la table finale Citiz AURA

Ce notebook prépare le CSV de release de la **table croisée du dashboard** : une ligne par station régionale, les indicateurs OSM et les attributs Citiz. Il ne recalcule pas les distances : exécuter d’abord les notebooks de préparation régionale si les entrées manquent.

Utiliser le noyau **Python (IGN / OSM)** (`ign-osm/.venv`). Exécuter toutes les cellules dans l’ordre depuis un noyau neuf. Aucun téléchargement n’est nécessaire. Les sources restent intactes ; une nouvelle exécution remplace les fichiers du dossier `regional-data/data/release/`.

Un CSV est un tableau texte : ici, séparateur virgule, décimales avec un point et encodage UTF-8 avec BOM pour faciliter son ouverture dans Excel. Les cellules vides représentent des valeurs inconnues, jamais des zéros ajoutés.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import shutil
import pandas as pd

# Locate the repository from its root or a notebook subdirectory.
root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "output/citiz_stations_geo.csv").exists()
             and (p / "regional-data").is_dir()), None)
if root is None:
    raise FileNotFoundError("Run this notebook inside the Citiz repository.")

feature_path = root / "regional-data/data/processed/station_features.csv"
station_path = root / "output/citiz_stations_geo.csv"
release_dir = root / "regional-data/data/release"
for path in [feature_path, station_path]:
    if not path.is_file():
        raise FileNotFoundError(f"Missing input: {path}")
print("Repository:", root)

Repository: /home/doryan-epitech/Desktop/Hackaton/ign-osm/Datz_Data-analyze-for-Citiz


## 1. Lire les tables et vérifier la clé

La **clé** `station_uid` identifie une station. Une **jointure** permet d’ajouter ses informations Citiz aux indicateurs OSM. Elle doit être de type un à un : un identifiant vide ou répété arrête l’export pour éviter de multiplier les lignes.

Le périmètre est celui de `station_features.csv`, déjà filtré avec le contour IGN AURA. On conserve toutes ses colonnes, notamment les statuts de qualité.

In [2]:
features = pd.read_csv(feature_path, low_memory=False)
stations = pd.read_csv(station_path, low_memory=False)
station_columns = [
    "station_uid", "city", "departement", "region", "reseau", "capacity",
    "num_vehicles_available", "is_renting", "extracted_at",
]
for name, frame in [("features", features), ("stations", stations)]:
    if frame["station_uid"].isna().any() or frame["station_uid"].astype(str).str.strip().eq("").any():
        raise ValueError(f"Empty station identifiers in {name}.")
    if not frame["station_uid"].is_unique:
        raise ValueError(f"Duplicate station identifiers in {name}.")
missing_columns = set(station_columns) - set(stations.columns)
if missing_columns:
    raise ValueError(f"Missing Citiz columns: {sorted(missing_columns)}")
print("Regional stations:", len(features))
print("National reference stations:", len(stations))

Regional stations: 612
National reference stations: 1614


## 2. Croiser les informations

La jointure **gauche** conserve exactement les stations de la table régionale. `station_reference_match` indique si leur référence Citiz a été retrouvée. Pour cette release, une référence manquante bloque l’export.

`capacity` désigne la capacité déclarée de la station Citiz. `num_vehicles_available` et `is_renting` viennent d’un instantané ; `extracted_at` est sa date d’extraction, pas une garantie de fraîcheur de chaque statut. Ces variables ne mesurent pas la demande ni l’utilisation historique.

In [3]:
final = features.merge(
    stations[station_columns], on="station_uid", how="left",
    validate="one_to_one", indicator="station_reference_match",
)
final["station_reference_match"] = final["station_reference_match"].astype(str)
if not final["station_reference_match"].eq("both").all():
    raise ValueError("Some regional stations have no Citiz reference.")
if final.empty or len(final) != len(features):
    raise ValueError("Unexpected output row count.")
if not (final["latitude"].between(-90, 90).all()
        and final["longitude"].between(-180, 180).all()):
    raise ValueError("Invalid or missing coordinates.")
# Verify that the join preserved every original feature, including nulls.
pd.testing.assert_frame_equal(final[features.columns], features.reset_index(drop=True))
final = final.sort_values("station_uid", kind="stable").reset_index(drop=True)
quality = pd.DataFrame({
    "column": final.columns,
    "dtype": [str(final[c].dtype) for c in final.columns],
    "missing_count": [int(final[c].isna().sum()) for c in final.columns],
    "missing_percent": [round(float(final[c].isna().mean() * 100), 2) for c in final.columns],
})
print(f"Final table: {len(final)} rows, {len(final.columns)} columns")
display(final.head())
display(quality[quality["missing_count"] > 0])

Final table: 612 rows, 50 columns


,station_uid,station_name,latitude,longitude,irve_count_500m,amenities_count_500m,amenities_count_500m_straight_line,amenities_unreachable_or_unsnapped_500m,bike_parking_count_300m,bike_parking_count_300m_straight_line,...,boundary_coverage_status,city,departement,region,reseau,capacity,num_vehicles_available,is_renting,extracted_at,station_reference_match
0,offre-et-disponibilite-temps-reel-du-service-c...,Lyon 2ème - Parking LPA République,45.760189,4.836602,NaN,56,323,177,5,36,...,within_region_conservative_check,NaN,Rhône,Auvergne-Rhône-Alpes,Métropole de Lyon,3,3,True,2026-10-19 14:43:54.580579+00:00,both
1,offre-et-disponibilite-temps-reel-du-service-c...,Lyon 6ème - Bugeaud - Garibaldi,45.766689,4.851268,NaN,1,326,0,4,61,...,within_region_conservative_check,NaN,Rhône,Auvergne-Rhône-Alpes,Métropole de Lyon,2,2,True,2026-10-19 14:43:54.580579+00:00,both
2,offre-et-disponibilite-temps-reel-du-service-c...,Sainte-Foy - Place St Luc,45.749405,4.801100,NaN,9,29,11,1,8,...,within_region_conservative_check,NaN,Rhône,Auvergne-Rhône-Alpes,Métropole de Lyon,1,1,True,2026-10-19 14:43:54.580579+00:00,both
3,offre-et-disponibilite-temps-reel-du-service-c...,Lyon 2ème - Charité - Auguste Comte,45.754833,4.831180,NaN,4,262,0,1,53,...,within_region_conservative_check,NaN,Rhône,Auvergne-Rhône-Alpes,Métropole de Lyon,2,1,True,2026-10-19 14:43:54.580579+00:00,both
4,offre-et-disponibilite-temps-reel-du-service-c...,Lyon 5ème - Quai Pierre Scize,45.766705,4.820361,NaN,0,42,0,1,25,...,within_region_conservative_check,NaN,Rhône,Auvergne-Rhône-Alpes,Métropole de Lyon,3,3,True,2026-10-19 14:43:54.580579+00:00,both


,column,dtype,missing_count,missing_percent
4,irve_count_500m,float64,612,100.00
11,transit_departures_peak_hour,float64,612,100.00
12,pedestrian_network_distance_m,float64,612,100.00
16,accidents_count_500m,float64,612,100.00
25,public_parking_spaces_300m,float64,555,90.69
41,city,float64,612,100.00


## 3. Exporter et relire le CSV

La relecture vérifie que les identifiants, les valeurs et les cellules manquantes ont survécu à l’écriture. Le manifeste enregistre une empreinte **SHA-256** des entrées et des sorties : elle permet de reconnaître exactement les fichiers utilisés.

Le dossier contient aussi le dictionnaire des indicateurs, les sources et un guide des attributs ajoutés. Il est exclu de Git comme les autres données du projet ; transmettre son contenu séparément pour la release.

In [4]:
release_dir.mkdir(parents=True, exist_ok=True)
csv_path = release_dir / "citiz_aura_final.csv"
quality_path = release_dir / "citiz_aura_quality.csv"
final.to_csv(csv_path, index=False, encoding="utf-8-sig", na_rep="", lineterminator="\n")
quality.to_csv(quality_path, index=False, encoding="utf-8-sig", lineterminator="\n")

reloaded = pd.read_csv(csv_path, low_memory=False)
pd.testing.assert_frame_equal(final, reloaded, check_dtype=False, check_exact=False,
                              rtol=1e-12, atol=1e-12)
assert final.isna().sum().equals(reloaded.isna().sum())
assert reloaded["station_uid"].is_unique

for filename in ["DATA_DICTIONARY.md", "SOURCES.md"]:
    shutil.copyfile(root / "regional-data" / filename, release_dir / filename)

release_readme = """# Table finale Citiz AURA

Une ligne par station du référentiel régional IGN, jointe au CSV Citiz national par station_uid.
Le fichier citiz_aura_final.csv utilise UTF-8 avec BOM, une virgule comme séparateur et un point décimal.
Les cellules vides sont inconnues ou non calculées. Aucun remplacement par zéro.

## Colonnes ajoutées au dictionnaire des indicateurs

- city : commune déclarée dans le référentiel Citiz ; peut être vide.
- departement, region : rattachement du référentiel Citiz ; le périmètre régional des stations a été confirmé en amont avec IGN.
- reseau : réseau déclaré dans le référentiel Citiz.
- capacity : capacité déclarée de station, pas le nombre de véhicules actuellement présents.
- num_vehicles_available : véhicules disponibles dans l'instantané Citiz.
- is_renting : statut de location déclaré dans cet instantané.
- extracted_at : date d'extraction du référentiel Citiz, pas date de mise à jour garantie de chaque station.
- station_reference_match : both signifie qu'une référence Citiz a été retrouvée.

## Interprétation et sources

Voir DATA_DICTIONARY.md pour les unités, rayons et statuts, et SOURCES.md pour les sources et licences.
Les indicateurs principaux suivent le réseau piéton OSM ; les suffixes _straight_line indiquent le vol d'oiseau.
Un zéro OSM décrit l'absence d'objet recensé dans les données traitées, pas une absence sur le terrain.
Les équipements de plusieurs stations peuvent se recouper : leur somme n'est pas un total régional unique.
Les sites OSM de recharge ne sont pas un nombre de bornes. La disponibilité instantanée ne mesure pas la demande.
La date d'export ne rend pas les observations plus récentes.
Attribution OSM : © OpenStreetMap contributors, ODbL 1.0, extraits Geofabrik.
Les métadonnées de licences IGN et GBFS restent à confirmer avant publication publique, comme indiqué dans SOURCES.md.

## Reproduction

Exécuter regional-data/notebooks/07_release_final_table.ipynb avec le noyau Python (IGN / OSM).
Le manifeste décrit les fichiers exacts par SHA-256. citiz_aura_quality.csv décrit les valeurs manquantes par colonne.
"""
(release_dir / "README.md").write_text(release_readme, encoding="utf-8")

def file_record(path):
    return {"path": path.relative_to(root).as_posix(), "bytes": path.stat().st_size,
            "sha256": hashlib.sha256(path.read_bytes()).hexdigest()}

manifest = {
    "exported_at_utc": datetime.now(timezone.utc).isoformat(),
    "rows": len(final), "columns": len(final.columns),
    "primary_key": "station_uid", "scope": "Auvergne-Rhone-Alpes (IGN regional station selection)",
    "csv": {"encoding": "utf-8-sig", "separator": ",", "decimal": ".", "missing_value": ""},
    "join": {"key": "station_uid", "how": "left", "validate": "one_to_one", "all_matched": True},
    "citiz_extracted_at": sorted(final["extracted_at"].dropna().unique().tolist()),
    "inputs": [file_record(p) for p in [feature_path, station_path]],
    "outputs": [file_record(release_dir / name) for name in [
        "citiz_aura_final.csv", "citiz_aura_quality.csv", "DATA_DICTIONARY.md", "SOURCES.md", "README.md"]],
    "versions": {"pandas": pd.__version__},
}
(release_dir / "release_manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("CSV verified:", csv_path)
print("Release directory:", release_dir)
print("Rows:", manifest["rows"], "Columns:", manifest["columns"])

CSV verified: /home/doryan-epitech/Desktop/Hackaton/ign-osm/Datz_Data-analyze-for-Citiz/regional-data/data/release/citiz_aura_final.csv
Release directory: /home/doryan-epitech/Desktop/Hackaton/ign-osm/Datz_Data-analyze-for-Citiz/regional-data/data/release
Rows: 612 Columns: 50


## Fichiers à transmettre

- `citiz_aura_final.csv` : tableau final pour l’analyse et les jointures.
- `citiz_aura_quality.csv` : types et valeurs manquantes par colonne.
- `release_manifest.json` : date d’export, contrôles et empreintes des fichiers.
- `DATA_DICTIONARY.md`, `SOURCES.md`, `README.md` : définitions, provenance et limites.

La préparation locale de ce dossier ne publie aucune donnée sur une plateforme externe.